# NB09 — Evaluation, Confidence Intervals, Videos & Summary

Rigorous evaluation of all methods (Random, Heuristic, PPO, SAC, Residual SAC)
on `UnitreeG1DishWipe-v1`.  Computes bootstrap confidence intervals, generates
comparison plots and logs everything to MLflow.


## Objective

1. Load all trained models (PPO, SAC, best Residual SAC).
2. Evaluate each for 100 episodes with deterministic actions.
3. Compute bootstrap 95% confidence intervals.
4. Generate comparison plots and video (if rendering available).
5. Compile final summary and log to MLflow.


## Environment

| Notebook | Goal | Required HW | Min CPU | Min RAM | GPU VRAM | Notes |
|---|---|---|---:|---:|---:|---|
| NB09 | Eval + CI + videos | CPU/GPU | 4 cores | 8 GB | 8-16 GB | GPU for speed |


In [ ]:
import sys, os, platform, json
print(f"Python: {sys.version}"); print(f"OS: {platform.platform()}")
import numpy as np; print(f"NumPy: {np.__version__}")
import torch; print(f"PyTorch: {torch.__version__}")
import gymnasium as gym; print(f"Gymnasium: {gym.__version__}")
import pandas as pd; print(f"Pandas: {pd.__version__}")


## Imports


In [ ]:
import json
import numpy as np
import torch
import gymnasium as gym
import pandas as pd
from pathlib import Path

# Ensure project root is on sys.path so `src.envs` can be imported
PROJECT_ROOT = Path(os.getcwd()).resolve()
if (PROJECT_ROOT / "src").is_dir():
    pass  # already at project root
elif (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError(f"Cannot find src/ from {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # normalize cwd for artifact paths

from src.envs.dishwipe_env import (
    UnitreeG1DishWipeEnv, CONTACT_THRESHOLD, PLATE_POS_IN_SINK,
)

## Config


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CFG = dict(
    seed=42,
    env_id="UnitreeG1DishWipe-v1",
    control_mode="pd_joint_delta_pos",
    obs_mode="state",
    eval_episodes=100,
    max_steps_per_ep=1000,
    bootstrap_samples=1000,
    ci_level=0.95,
    device=DEVICE,
)
SEED = CFG["seed"]
artifact_dir = Path("artifacts/NB09")
artifact_dir.mkdir(parents=True, exist_ok=True)
print("Config:", json.dumps(CFG, indent=2))


## Reproducibility


In [ ]:
import random; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"✅ Seeds set to {SEED}")


## Implementation Steps

### Step 1 — Load models


In [ ]:
from stable_baselines3 import PPO, SAC

models = {}

# PPO
ppo_path = Path("artifacts/NB06/ppo_model.zip")
if ppo_path.exists():
    models["PPO"] = PPO.load(str(ppo_path), device=DEVICE)
    print(f"✅ Loaded PPO from {ppo_path}")
else:
    print(f"⚠️ PPO model not found at {ppo_path}")

# SAC
sac_path = Path("artifacts/NB07/sac_model.zip")
if sac_path.exists():
    models["SAC"] = SAC.load(str(sac_path), device=DEVICE)
    print(f"✅ Loaded SAC from {sac_path}")
else:
    print(f"⚠️ SAC model not found at {sac_path}")

# Residual SAC (best beta)
ablation_path = Path("artifacts/NB08/ablation_beta_table.csv")
if ablation_path.exists():
    abl_df = pd.read_csv(ablation_path)
    best_beta = abl_df.loc[abl_df["mean_reward"].idxmax(), "beta"]
    res_path = Path(f"artifacts/NB08/residual_sac_beta{best_beta}.zip")
    if res_path.exists():
        models[f"Residual SAC (β={best_beta})"] = SAC.load(str(res_path), device=DEVICE)
        print(f"✅ Loaded Residual SAC β={best_beta}")
    else:
        print(f"⚠️ Residual SAC model not found: {res_path}")
else:
    best_beta = None
    print("⚠️ Ablation table not found — skipping Residual SAC")

print(f"\nLoaded {len(models)} trained models.")


### Step 2 — Define evaluation loop


In [ ]:
def evaluate_model(env, model_or_fn, n_episodes, max_steps, seed,
                   is_residual=False, base_controller=None, beta=1.0):
    """Evaluate a model or function-based policy."""
    results = dict(
        rewards=[], cleaned_ratios=[], steps=[], mean_jerks=[],
        mean_forces=[], contact_rates=[], successes=[],
    )

    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        if base_controller:
            base_controller.reset()
        total_rew = 0.0
        prev_act = np.zeros(env.action_space.shape[-1])
        jerks, forces, contacts = [], [], 0

        for step in range(max_steps):
            if callable(model_or_fn) and not hasattr(model_or_fn, "predict"):
                action = model_or_fn(obs, env)
            elif is_residual and base_controller:
                residual, _ = model_or_fn.predict(obs, deterministic=True)
                if isinstance(residual, torch.Tensor):
                    residual = residual.cpu().numpy()
                a_base = base_controller(obs)
                action = np.clip(a_base + beta * residual,
                                env.action_space.low, env.action_space.high)
            else:
                action, _ = model_or_fn.predict(obs, deterministic=True)

            if isinstance(action, torch.Tensor):
                action_np = action.cpu().numpy()
            else:
                action_np = np.array(action, dtype=np.float32)

            obs, reward, terminated, truncated, info = env.step(action)
            total_rew += reward.item() if hasattr(reward, "item") else float(reward)

            jerk = float(np.sum((action_np.flatten() - prev_act.flatten()) ** 2))
            jerks.append(jerk)
            prev_act = action_np.flatten().copy()

            fz = info.get("contact_force", torch.tensor([0.0]))
            fz_val = fz.item() if hasattr(fz, "item") else float(fz)
            forces.append(fz_val)
            if fz_val >= CONTACT_THRESHOLD:
                contacts += 1

            if (terminated.any() if hasattr(terminated, "any") else terminated):
                break

        ratio = info.get("cleaned_ratio", torch.tensor([0.0]))
        r = ratio.item() if hasattr(ratio, "item") else float(ratio)
        n_steps = step + 1
        success = info.get("success", torch.tensor([False]))
        s = bool(success.item() if hasattr(success, "item") else success)

        results["rewards"].append(total_rew)
        results["cleaned_ratios"].append(r)
        results["steps"].append(n_steps)
        results["mean_jerks"].append(np.mean(jerks))
        results["mean_forces"].append(np.mean(forces))
        results["contact_rates"].append(contacts / n_steps)
        results["successes"].append(s)

    return results

print("✅ evaluate_model() defined.")


### Step 3 — Evaluate all methods


In [ ]:
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

eval_env = gym.make(
    CFG["env_id"], obs_mode=CFG["obs_mode"],
    control_mode=CFG["control_mode"], num_envs=1, render_mode=None,
)
eval_env = CPUGymWrapper(eval_env)

all_results = {}

# 1. Random baseline
def random_policy(obs, env_):
    return env_.action_space.sample()

print("Evaluating Random...")
all_results["Random"] = evaluate_model(
    eval_env, random_policy, min(CFG["eval_episodes"], 20),
    CFG["max_steps_per_ep"], SEED
)

# 2. Heuristic (uses palm position, aligned with env v2 reach reward)
def heuristic_policy_eval(obs, env_):
    unwrapped = env_.unwrapped if hasattr(env_, "unwrapped") else env_
    try:
        palm = unwrapped.agent.robot.links_map["left_palm_link"].pose.p[0].cpu().numpy()
        plate = unwrapped.plate.pose.p[0].cpu().numpy()
        delta = plate - palm
        act = np.zeros(env_.action_space.shape[-1], dtype=np.float32)
        for idx in range(1, 4):
            act[idx] = np.clip(delta[idx % 3] * 0.5, -0.3, 0.3)
        return act
    except Exception:
        return np.zeros(env_.action_space.shape[-1], dtype=np.float32)

print("Evaluating Heuristic...")
all_results["Heuristic"] = evaluate_model(
    eval_env, heuristic_policy_eval, min(CFG["eval_episodes"], 20),
    CFG["max_steps_per_ep"], SEED
)

# 3. Trained models
for name, model in models.items():
    print(f"Evaluating {name}...")
    if "Residual" in name and best_beta:
        base_ctrl_eval = None
        try:
            # Quick inline base controller
            class _BC:
                def __init__(self, env_):
                    self.env = env_; self.alpha = 0.3
                    self._prev = None; self.act_dim = env_.action_space.shape[-1]

                def reset(self):
                    self._prev = np.zeros(self.act_dim, dtype=np.float32)

                def __call__(self, obs):
                    raw = heuristic_policy_eval(obs, self.env)
                    if self._prev is None: self._prev = np.zeros_like(raw)
                    s = self.alpha * raw + (1 - self.alpha) * self._prev
                    self._prev = s.copy()
                    return s

            base_ctrl_eval = _BC(eval_env)
        except Exception:
            pass

        all_results[name] = evaluate_model(
            eval_env, model, CFG["eval_episodes"], CFG["max_steps_per_ep"],
            SEED, is_residual=True, base_controller=base_ctrl_eval,
            beta=best_beta,
        )
    else:
        all_results[name] = evaluate_model(
            eval_env, model, CFG["eval_episodes"], CFG["max_steps_per_ep"], SEED
        )

print(f"\n✅ Evaluated {len(all_results)} methods.")



### Step 4 — Bootstrap confidence intervals


In [ ]:
def bootstrap_ci(data, n_boot=1000, ci=0.95):
    """Compute bootstrap confidence interval for the mean."""
    data = np.array(data)
    boot_means = []
    for _ in range(n_boot):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(sample))
    boot_means = np.sort(boot_means)
    lo = boot_means[int((1 - ci) / 2 * n_boot)]
    hi = boot_means[int((1 + ci) / 2 * n_boot)]
    return float(np.mean(data)), float(lo), float(hi)


summary_rows = []
for name, res in all_results.items():
    row = {"Method": name}
    for metric_key, display in [
        ("rewards", "Reward"),
        ("cleaned_ratios", "Cleaned"),
        ("mean_jerks", "Jerk"),
        ("contact_rates", "Contact Rate"),
    ]:
        mean, lo, hi = bootstrap_ci(
            res[metric_key], CFG["bootstrap_samples"], CFG["ci_level"]
        )
        row[f"{display} (mean)"] = f"{mean:.4f}"
        row[f"{display} (95% CI)"] = f"[{lo:.4f}, {hi:.4f}]"

    success_rate = np.mean(res["successes"]) if res["successes"] else 0.0
    row["Success Rate"] = f"{success_rate*100:.1f}%"
    summary_rows.append(row)

df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

eval_table_path = artifact_dir / "eval_table.csv"
df.to_csv(eval_table_path, index=False)
print(f"\n✅ Saved: {eval_table_path}")


### Step 5 — Comparison plots


In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

methods = list(all_results.keys())
n_methods = len(methods)
colors = plt.cm.Set2(np.linspace(0, 1, max(n_methods, 3)))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Reward
ax = axes[0]
means = [np.mean(all_results[m]["rewards"]) for m in methods]
stds = [np.std(all_results[m]["rewards"]) for m in methods]
ax.bar(range(n_methods), means, yerr=stds, color=colors[:n_methods], capsize=3)
ax.set_xticks(range(n_methods)); ax.set_xticklabels(methods, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean Episode Reward"); ax.set_title("Reward")

# Cleaned ratio
ax = axes[1]
means = [np.mean(all_results[m]["cleaned_ratios"]) for m in methods]
stds = [np.std(all_results[m]["cleaned_ratios"]) for m in methods]
ax.bar(range(n_methods), means, yerr=stds, color=colors[:n_methods], capsize=3)
ax.set_xticks(range(n_methods)); ax.set_xticklabels(methods, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean Cleaned Ratio"); ax.set_title("Cleaning Performance")

# Jerk
ax = axes[2]
means = [np.mean(all_results[m]["mean_jerks"]) for m in methods]
stds = [np.std(all_results[m]["mean_jerks"]) for m in methods]
ax.bar(range(n_methods), means, yerr=stds, color=colors[:n_methods], capsize=3)
ax.set_xticks(range(n_methods)); ax.set_xticklabels(methods, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean Jerk"); ax.set_title("Action Smoothness")

fig.suptitle("NB09 — Method Comparison (UnitreeG1DishWipe-v1)", fontsize=14)
fig.tight_layout()
fig.savefig(str(artifact_dir / "eval_comparison.png"), dpi=150, bbox_inches="tight")
plt.close("all")
print("✅ Saved: eval_comparison.png")


### Step 6 — Render video (optional)


In [ ]:
# Attempt to render one episode of the best method
try:
    best_method = max(all_results.keys(),
                     key=lambda m: np.mean(all_results[m]["cleaned_ratios"]))
    print(f"Best method for video: {best_method}")

    vid_env = gym.make(
        CFG["env_id"], obs_mode=CFG["obs_mode"],
        control_mode=CFG["control_mode"], num_envs=1,
        render_mode="rgb_array",
    )
    vid_env = CPUGymWrapper(vid_env)
    obs, _ = vid_env.reset(seed=SEED)
    frames = []
    for step in range(200):
        if best_method in models:
            action, _ = models[best_method].predict(obs, deterministic=True)
        else:
            action = vid_env.action_space.sample()
        obs, _, terminated, truncated, info = vid_env.step(action)
        try:
            frame = vid_env.render()
            if frame is not None:
                frames.append(frame)
        except Exception:
            pass
        if (terminated.any() if hasattr(terminated, "any") else terminated):
            break

    vid_env.close()

    if frames:
        import imageio
        video_path = artifact_dir / "best_method_video.mp4"
        imageio.mimsave(str(video_path), frames, fps=30)
        print(f"✅ Video saved: {video_path}")
    else:
        print("⚠️ No frames captured (rendering may not be available)")
except Exception as e:
    print(f"⚠️ Video generation failed: {e}")
    print("   (This is expected on CPU-only machines without Vulkan)")


### Step 7 — MLflow summary


In [ ]:
try:
    import mlflow; from dotenv import load_dotenv; load_dotenv(".env.local")
    uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if uri:
        mlflow.set_tracking_uri(uri)
        mlflow.set_experiment("dishwipe_unitree_g1")
        with mlflow.start_run(run_name="NB09_final_summary"):
            mlflow.log_params(CFG)
            for name, res in all_results.items():
                tag = name.replace(" ", "_").replace("(","").replace(")","").replace("=","")
                mlflow.log_metric(f"{tag}_reward", float(np.mean(res["rewards"])))
                mlflow.log_metric(f"{tag}_cleaned", float(np.mean(res["cleaned_ratios"])))
                mlflow.log_metric(f"{tag}_jerk", float(np.mean(res["mean_jerks"])))
                mlflow.log_metric(f"{tag}_success", float(np.mean(res["successes"])))
            mlflow.log_artifacts(str(artifact_dir), artifact_path="NB09")
        print("✅ MLflow final summary logged.")
except Exception as e:
    print(f"⚠️ MLflow: {e}")


## Results

**Evaluation Summary:**
- All methods evaluated on `UnitreeG1DishWipe-v1` with deterministic actions
- Bootstrap 95% CI computed over evaluation episodes
- Comparison plots show reward, cleaning performance, and action smoothness
- Video recorded for best-performing method (if rendering available)


## Artifacts

| File | Description |
|------|-------------|
| `artifacts/NB09/eval_table.csv` | Full evaluation table with CIs |
| `artifacts/NB09/eval_comparison.png` | Bar chart comparison |
| `artifacts/NB09/best_method_video.mp4` | Video (optional) |


## Cleanup


In [ ]:
eval_env.close()
print('✅ NB09 complete. Pipeline finished!')


## References

- Efron & Tibshirani (1993) — Bootstrap methods for CIs
- Schulman et al. (2017) — PPO
- Haarnoja et al. (2018) — SAC
- Silver et al. (2018) — Residual Policy Learning
- ManiSkill3: https://maniskill.readthedocs.io/
